# EMCRAG — benchmark standalone M0 → M1

Eseguire in **VS Code / WSL Ubuntu 22.04**, profilo **EMCRAG**, kernel **Python (conda_ok_14apr)**.

Questo notebook:
- usa i 196 input congelati;
- usa le funzioni di generazione e pulizia estratte dal notebook v06.11;
- esegue M0 e M1 con lo stesso codice e lo stesso contratto;
- salva ogni risposta immediatamente e riprende dopo interruzioni;
- carica un solo modello alla volta sulla GPU.

Eseguire le celle in ordine. Non modificare configurazione o prompt dopo l’avvio del benchmark.

In [ ]:
from pathlib import Path
from collections import Counter
import contextlib, gc, hashlib, importlib.metadata as mdlib
import io, json, os, platform, random, sys, time
import numpy as np
import torch
from PIL import Image

PROJECT_DIR = Path("/home/eugrug/Project_CRAG_MedGemma")
INPUT_DIR = PROJECT_DIR / "benchmark_m0_m1" / "inputs_196"
MANIFEST_PATH = INPUT_DIR / "EMCRAG_manifest_196.jsonl"
OUTPUT_DIR = PROJECT_DIR / "benchmark_m0_m1" / "outputs"
CACHE_DIR = Path("/mnt/f/hf_home_cache")

BASE_MODEL = "google/medgemma-1.5-4b-it"
ADAPTER_DIR = Path("/home/eugrug/.unsloth/studio/outputs/google_medgemma-1.5-4b-it__project-emcrag_ft_imb_010k_googlebase_v0702_1788538483")
EXPECTED_MANIFEST_SHA256 = "51fe2c0d201bf00a9bbed368195009d4ad72edbe740f146179cf04694389c1b5"
EXPECTED_ADAPTER_SHA256 = "e64c995b6f3b42be3d9ff19c85d83c688cc6458673024dc9bcef0bf95d241098"

SYSTEM_PROMPT = (
    "Sei un assistente medico che risponde in lingua italiana. "
    "Fornisci una risposta pertinente e chiara alla domanda dell'utente."
)
SEED = 20260831

def sha_bytes(data):
    return hashlib.sha256(data).hexdigest()

def sha_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def require(ok, message):
    if not ok:
        raise RuntimeError(message)

require(MANIFEST_PATH.is_file(), f"Manifest assente: {MANIFEST_PATH}")
require(ADAPTER_DIR.is_dir(), f"Adapter assente: {ADAPTER_DIR}")
require(CACHE_DIR.is_dir(), f"Cache assente: {CACHE_DIR}")
require(sha_file(MANIFEST_PATH) == EXPECTED_MANIFEST_SHA256, "Checksum manifest diverso: STOP")
require(sha_file(ADAPTER_DIR / "adapter_model.safetensors") == EXPECTED_ADAPTER_SHA256, "Checksum adapter diverso: STOP")

rows = [json.loads(line) for line in MANIFEST_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
require(len(rows) == len({r["sample_id"] for r in rows}) == 196, "Manifest non valido")
require(Counter(r["dataset"] for r in rows) == {"IMB_Curated_Test_Set_1000": 100, "MMMED_Italian": 96}, "Coorte inattesa")
require([r["benchmark_order"] for r in rows] == list(range(196)), "Ordine benchmark non valido")
for r in rows:
    if r["image_required"]:
        p = INPUT_DIR / r["image_path"]
        require(p.is_file() and sha_file(p) == r["image_sha256"], f"Immagine non valida: {r['sample_id']}")

v0611_candidates = [
    PROJECT_DIR / "CRAG_MedGemma_v.06.11.ipynb",
    *sorted(PROJECT_DIR.glob("CRAG_MedGemma_v.06.11*.ipynb")),
]
v0611_candidates = list(dict.fromkeys(p for p in v0611_candidates if p.is_file()))
require(len(v0611_candidates) >= 1, "Notebook CRAG_MedGemma_v.06.11 non trovato nella directory del progetto")
V0611_NOTEBOOK = v0611_candidates[0]

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

print("Python:", sys.executable)
print("PyTorch:", torch.__version__, "| CUDA:", torch.version.cuda, "| GPU:", torch.cuda.get_device_name(0))
print("Manifest:", len(rows), "record | SHA256:", EXPECTED_MANIFEST_SHA256)
print("Adapter SHA256:", EXPECTED_ADAPTER_SHA256)
print("Sorgente v06.11:", V0611_NOTEBOOK)


## Importazione controllata delle funzioni v06.11

La cella identifica la singola cella del notebook congelato che contiene sia `generate_response()` sia `postprocess_medgemma_answer()`, ne calcola l'hash e la esegue nel namespace corrente.

In [ ]:
DEBUG_MODE = False

def free_vram_resources():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass

class _SilentMonitor:
    @staticmethod
    def show(*args, **kwargs):
        return None

Alienware_Monitor = _SilentMonitor()

v0611_json = json.loads(V0611_NOTEBOOK.read_text(encoding="utf-8"))
generator_cells = []
for cell in v0611_json["cells"]:
    source = "".join(cell.get("source", []))
    if "def generate_response(" in source and "def postprocess_medgemma_answer(" in source:
        generator_cells.append(source)

require(len(generator_cells) == 1, f"Attesa una cella generatore v06.11; trovate: {len(generator_cells)}")
V0611_GENERATOR_SOURCE = generator_cells[0]
V0611_GENERATOR_SOURCE_SHA256 = sha_bytes(V0611_GENERATOR_SOURCE.encode("utf-8"))
exec(compile(V0611_GENERATOR_SOURCE, str(V0611_NOTEBOOK) + "::generator_cell", "exec"), globals())

for name in ("generate_response", "postprocess_medgemma_answer", "is_ragas_eligible", "detect_generation_leakage"):
    require(callable(globals().get(name)), f"Funzione v06.11 non caricata: {name}")

print("Funzioni v06.11 caricate.")
print("Generator source SHA256:", V0611_GENERATOR_SOURCE_SHA256)


## Contratto, caricamento e runner

M0 usa il repository Google base. M1 usa l'adapter FT-10K congelato, che dichiara la stessa base. Il processor viene sempre caricato dal repository base.

In [ ]:
from unsloth import FastLanguageModel
from transformers import AutoProcessor

adapter_config = json.loads((ADAPTER_DIR / "adapter_config.json").read_text(encoding="utf-8"))
require(adapter_config.get("base_model_name_or_path") == BASE_MODEL, "Base model dell'adapter diversa: STOP")

CONTRACT = {
    "contract_version": "EMCRAG_M0_M1_v07.03",
    "cohort_records": 196,
    "manifest_sha256": EXPECTED_MANIFEST_SHA256,
    "base_model": BASE_MODEL,
    "adapter_path": str(ADAPTER_DIR),
    "adapter_sha256": EXPECTED_ADAPTER_SHA256,
    "v0611_notebook": str(V0611_NOTEBOOK),
    "v0611_notebook_sha256": sha_file(V0611_NOTEBOOK),
    "v0611_generator_source_sha256": V0611_GENERATOR_SOURCE_SHA256,
    "system_prompt": SYSTEM_PROMPT,
    "context_window": 2048,
    "max_new_tokens": 512,
    "min_new_tokens": 128,
    "safety_margin": 96,
    "do_sample": False,
    "temperature": None,
    "top_p": None,
    "top_k": None,
    "repetition_penalty": 1.10,
    "seed": SEED,
    "image_policy": "Frozen current bytes shared identically by M0 and M1.",
    "historical_R0_image_byte_identity": "Unavailable from retained R0 archive.",
}
CONTRACT_BYTES = (json.dumps(CONTRACT, ensure_ascii=False, sort_keys=True, indent=2) + "\n").encode()
CONTRACT_SHA256 = sha_bytes(CONTRACT_BYTES)
contract_path = OUTPUT_DIR / "inference_contract.json"
if contract_path.exists():
    require(contract_path.read_bytes() == CONTRACT_BYTES, "Contratto esistente diverso: STOP")
else:
    contract_path.write_bytes(CONTRACT_BYTES)

medgemma_model = None
medgemma_tokenizer = None
medgemma_processor = None
active_condition = None

def seed_everything():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

def unload_model():
    global medgemma_model, medgemma_tokenizer, medgemma_processor, active_condition
    medgemma_model = None
    medgemma_tokenizer = None
    medgemma_processor = None
    active_condition = None
    free_vram_resources()
    if torch.cuda.is_available():
        print(f"VRAM dopo unload: {torch.cuda.memory_allocated()/2**30:.2f} GiB allocata")

def load_condition(condition):
    global medgemma_model, medgemma_tokenizer, medgemma_processor, active_condition
    require(condition in {"M0", "M1"}, "Condizione non valida")
    unload_model()
    seed_everything()
    model_source = BASE_MODEL if condition == "M0" else str(ADAPTER_DIR)
    print(f"Caricamento {condition}: {model_source}")
    medgemma_model, medgemma_tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_source,
        max_seq_length=2048,
        dtype=None,
        load_in_4bit=True,
        cache_dir=str(CACHE_DIR),
    )
    medgemma_processor = AutoProcessor.from_pretrained(BASE_MODEL, cache_dir=str(CACHE_DIR))
    inner = getattr(medgemma_tokenizer, "tokenizer", medgemma_tokenizer)
    inner.pad_token = inner.eos_token
    inner.padding_side = "right"
    FastLanguageModel.for_inference(medgemma_model)
    active_condition = condition
    print(f"{condition} caricato | VRAM: {torch.cuda.memory_allocated()/2**30:.2f} GiB")

def build_messages(question, with_image):
    user_content = (
        [{"type": "image"}, {"type": "text", "text": question}]
        if with_image else question
    )
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
    ]

def build_prompt(question, with_image):
    return medgemma_processor.apply_chat_template(
        build_messages(question, with_image),
        tokenize=False,
        add_generation_prompt=True,
    )

def open_record_image(record):
    if not record["image_required"]:
        return None
    path = INPUT_DIR / record["image_path"]
    with Image.open(path) as im:
        return im.convert("RGB").copy()

def input_token_count(prompt, image):
    if image is None:
        inputs = medgemma_tokenizer(text=prompt, return_tensors="pt", add_special_tokens=True)
    else:
        text = prompt if "<image>" in prompt else "<image>\n" + prompt
        inputs = medgemma_processor(text=text, images=image, return_tensors="pt", add_special_tokens=True)
    return int(inputs.input_ids.shape[1])

def smoke_test():
    require(active_condition in {"M0", "M1"}, "Caricare prima il modello")
    cases = [
        ("Rispondi in una frase: che cos'è la febbre?", None),
        ("Descrivi prudentemente questa immagine tecnica uniforme.", Image.new("RGB", (64, 64), "white")),
    ]
    for question, image in cases:
        prompt = build_prompt(question, image is not None)
        answer = generate_response(prompt, image=image)
        require(bool(answer.strip()), f"Smoke test {active_condition} vuoto")
        print(active_condition, "smoke", "multimodal" if image else "text", "OK |", answer[:160].replace("\n", " "))
    print(active_condition, "SMOKE PASS")

def load_completed(path, condition):
    completed = {}
    if not path.exists():
        return completed
    for line in path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        rec = json.loads(line)
        require(rec.get("condition") == condition, f"Condizione errata nel file {path}")
        require(rec.get("contract_sha256") == CONTRACT_SHA256, f"Contratto diverso nel file {path}: STOP")
        if rec.get("status") == "complete":
            completed[rec["sample_id"]] = rec
    return completed

def run_condition(condition):
    require(active_condition == condition, f"Caricare {condition} prima dell'esecuzione")
    out_path = OUTPUT_DIR / f"{condition}_outputs.jsonl"
    completed = load_completed(out_path, condition)
    print(f"{condition}: {len(completed)}/196 già completi")
    for position, record in enumerate(rows, 1):
        sid = record["sample_id"]
        if sid in completed:
            continue
        image = None
        started = time.perf_counter()
        result = {
            "benchmark_order": record["benchmark_order"],
            "sample_id": sid,
            "dataset": record["dataset"],
            "execution_mode": record["execution_mode"],
            "condition": condition,
            "question": record["question"],
            "reference_answer": record["reference_answer"],
            "correct_letter": record.get("correct_letter"),
            "options": record.get("options"),
            "image_path": record.get("image_path"),
            "image_sha256": record.get("image_sha256"),
            "contract_sha256": CONTRACT_SHA256,
            "status": "error",
            "exception": None,
        }
        try:
            image = open_record_image(record)
            prompt = build_prompt(record["question"], image is not None)
            result["prompt_sha256"] = sha_bytes(prompt.encode("utf-8"))
            result["input_tokens"] = input_token_count(prompt, image)
            result["effective_max_new_tokens"] = min(512, 2048 - result["input_tokens"] - 96)
            require(result["effective_max_new_tokens"] >= 128, f"Budget token insufficiente: {sid}")
            capture = io.StringIO()
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            gen_started = time.perf_counter()
            with contextlib.redirect_stdout(capture):
                raw_answer = generate_response(prompt, image=image)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            cleaned, applied, notes = postprocess_medgemma_answer(raw_answer)
            eligible, artifacts, eligibility_notes = is_ragas_eligible(cleaned)
            inner = getattr(medgemma_tokenizer, "tokenizer", medgemma_tokenizer)
            result.update({
                "raw_generated_response": raw_answer,
                "answer_for_evaluation": cleaned,
                "post_filter_applied": applied,
                "post_filter_notes": notes,
                "eligible": eligible,
                "artifacts": artifacts,
                "eligibility_notes": eligibility_notes,
                "reasoning_leakage_detected": detect_generation_leakage(raw_answer),
                "output_tokens": len(inner.encode(raw_answer, add_special_tokens=False)),
                "generation_latency_sec": round(time.perf_counter() - gen_started, 6),
                "status": "complete",
            })
        except Exception as exc:
            result["exception"] = f"{type(exc).__name__}: {exc}"
        result["total_latency_sec"] = round(time.perf_counter() - started, 6)
        with out_path.open("a", encoding="utf-8") as f:
            f.write(json.dumps(result, ensure_ascii=False, sort_keys=True) + "\n")
            f.flush()
            os.fsync(f.fileno())
        print(f"[{position}/196] {condition} {sid}: {result['status']} | {result['total_latency_sec']:.1f}s", flush=True)
        if result["status"] != "complete":
            raise RuntimeError(f"{condition} interrotto su {sid}: {result['exception']}. Correggere e rilanciare la stessa cella.")
        completed[sid] = result
    print(f"{condition} PASS: 196/196 completi")
    return out_path

print("Contratto SHA256:", CONTRACT_SHA256)
print("Funzioni runner pronte.")


## M0 — caricamento e smoke test

In [ ]:
load_condition("M0")
smoke_test()


## M0 — benchmark 196 casi

La cella salva ogni record. Se l'esecuzione si interrompe, rilanciare questa stessa cella: i record completi vengono saltati.

In [ ]:
M0_OUTPUT_PATH = run_condition("M0")


## Rilascio M0 dalla GPU

In [ ]:
unload_model()


## M1 — caricamento e smoke test

In [ ]:
load_condition("M1")
smoke_test()


## M1 — benchmark 196 casi

In [ ]:
M1_OUTPUT_PATH = run_condition("M1")


## Completion gate e dataset paired

In [ ]:
def latest_complete(path, condition):
    records = load_completed(path, condition)
    require(len(records) == 196, f"{condition}: completi {len(records)}/196")
    return [records[r["sample_id"]] for r in rows]

m0 = latest_complete(OUTPUT_DIR / "M0_outputs.jsonl", "M0")
m1 = latest_complete(OUTPUT_DIR / "M1_outputs.jsonl", "M1")
require([r["sample_id"] for r in m0] == [r["sample_id"] for r in m1] == [r["sample_id"] for r in rows], "Ordine paired diverso")
require(all(a["prompt_sha256"] == b["prompt_sha256"] for a, b in zip(m0, m1)), "Prompt M0/M1 diversi: STOP")
require(all(a.get("image_sha256") == b.get("image_sha256") for a, b in zip(m0, m1)), "Immagini M0/M1 diverse: STOP")

paired = []
for source, a, b in zip(rows, m0, m1):
    paired.append({
        "benchmark_order": source["benchmark_order"],
        "sample_id": source["sample_id"],
        "dataset": source["dataset"],
        "execution_mode": source["execution_mode"],
        "question": source["question"],
        "reference_answer": source["reference_answer"],
        "correct_letter": source.get("correct_letter"),
        "options": source.get("options"),
        "prompt_sha256": a["prompt_sha256"],
        "image_sha256": source.get("image_sha256"),
        "M0_raw_generated_response": a["raw_generated_response"],
        "M0_answer_for_evaluation": a["answer_for_evaluation"],
        "M0_eligible": a["eligible"],
        "M0_latency_sec": a["generation_latency_sec"],
        "M1_raw_generated_response": b["raw_generated_response"],
        "M1_answer_for_evaluation": b["answer_for_evaluation"],
        "M1_eligible": b["eligible"],
        "M1_latency_sec": b["generation_latency_sec"],
        "contract_sha256": CONTRACT_SHA256,
    })

paired_bytes = "".join(json.dumps(r, ensure_ascii=False, sort_keys=True) + "\n" for r in paired).encode()
paired_path = OUTPUT_DIR / "M0_M1_paired_196.jsonl"
if paired_path.exists():
    require(paired_path.read_bytes() == paired_bytes, "Dataset paired esistente diverso: STOP")
else:
    paired_path.write_bytes(paired_bytes)

summary = {
    "status": "PASS",
    "M0_complete": 196,
    "M1_complete": 196,
    "paired_complete": 196,
    "prompt_identity": True,
    "image_identity": True,
    "M0_eligible": sum(bool(r["eligible"]) for r in m0),
    "M1_eligible": sum(bool(r["eligible"]) for r in m1),
    "manifest_sha256": EXPECTED_MANIFEST_SHA256,
    "contract_sha256": CONTRACT_SHA256,
    "paired_sha256": sha_bytes(paired_bytes),
}
summary_path = OUTPUT_DIR / "M0_M1_completion_gate.json"
summary_bytes = (json.dumps(summary, ensure_ascii=False, sort_keys=True, indent=2) + "\n").encode()
if summary_path.exists():
    require(summary_path.read_bytes() == summary_bytes, "Completion gate esistente diverso: STOP")
else:
    summary_path.write_bytes(summary_bytes)

print(json.dumps(summary, ensure_ascii=False, indent=2))
print("Paired output:", paired_path)


## FT-10K — congelamento e provenienza del modello

Verifica conclusiva del run definitivo Google-base: validation loss,
checkpoint, configurazioni, revisione del modello base e checksum degli artefatti.

In [1]:
from pathlib import Path
import json

ADAPTER_DIR = Path(
    "/home/eugrug/.unsloth/studio/outputs/"
    "google_medgemma-1.5-4b-it__project-emcrag_ft_imb_010k_googlebase_v0702_1788538483"
)

STATE_PATH = ADAPTER_DIR / "checkpoint-1245" / "trainer_state.json"

state = json.loads(STATE_PATH.read_text(encoding="utf-8"))

eval_records = [
    record
    for record in state.get("log_history", [])
    if "eval_loss" in record
]

print("Trainer state:", STATE_PATH)
print("Global step:", state.get("global_step"))
print("Best metric:", state.get("best_metric"))
print("Best model checkpoint:", state.get("best_model_checkpoint"))
print("Numero valutazioni:", len(eval_records))

print("\nTutte le validation loss:")
for record in eval_records:
    print({
        "step": record.get("step"),
        "epoch": record.get("epoch"),
        "eval_loss": record.get("eval_loss"),
    })

print("\nValutazione finale registrata:")
print(eval_records[-1] if eval_records else "NESSUNA EVAL_LOSS TROVATA")

Trainer state: /home/eugrug/.unsloth/studio/outputs/google_medgemma-1.5-4b-it__project-emcrag_ft_imb_010k_googlebase_v0702_1788538483/checkpoint-1245/trainer_state.json
Global step: 1245
Best metric: None
Best model checkpoint: None
Numero valutazioni: 4

Tutte le validation loss:
{'step': 312, 'epoch': 0.25060240963855424, 'eval_loss': 1.8491053581237793}
{'step': 624, 'epoch': 0.5012048192771085, 'eval_loss': 1.800355315208435}
{'step': 936, 'epoch': 0.7518072289156627, 'eval_loss': 1.769695520401001}
{'step': 1245, 'epoch': 1.0, 'eval_loss': 1.7569763660430908}

Valutazione finale registrata:
{'epoch': 1.0, 'eval_loss': 1.7569763660430908, 'eval_runtime': 1142.3904, 'eval_samples_per_second': 1.741, 'eval_steps_per_second': 1.741, 'num_input_tokens_seen': 3893306, 'step': 1245}


In [2]:
from pathlib import Path
import hashlib
import json

ADAPTER_DIR = Path(
    "/home/eugrug/.unsloth/studio/outputs/"
    "google_medgemma-1.5-4b-it__project-emcrag_ft_imb_010k_googlebase_v0702_1788538483"
)

CACHE_ROOTS = [
    Path("/mnt/f/hf_home_cache"),
    Path.home() / ".cache" / "huggingface",
]

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

adapter_config_path = ADAPTER_DIR / "adapter_config.json"
adapter_config = json.loads(adapter_config_path.read_text(encoding="utf-8"))

print("BASE MODEL:", adapter_config.get("base_model_name_or_path"))
print("REVISION IN ADAPTER CONFIG:", adapter_config.get("revision"))

print("\nFILE CONGELATI:")
wanted_names = {
    "adapter_model.safetensors",
    "adapter_config.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "special_tokens_map.json",
    "processor_config.json",
    "preprocessor_config.json",
    "chat_template.jinja",
}

for path in sorted(ADAPTER_DIR.iterdir()):
    if path.is_file() and path.name in wanted_names:
        print(path.name, path.stat().st_size, sha256_file(path))

state_path = ADAPTER_DIR / "checkpoint-1245" / "trainer_state.json"
print(
    "checkpoint-1245/trainer_state.json",
    state_path.stat().st_size,
    sha256_file(state_path),
)

print("\nSNAPSHOT LOCALI DEL MODELLO BASE:")
found = []
for root in CACHE_ROOTS:
    for relative in [
        Path("hub/models--google--medgemma-1.5-4b-it/snapshots"),
        Path("models--google--medgemma-1.5-4b-it/snapshots"),
    ]:
        snapshots_dir = root / relative
        if snapshots_dir.is_dir():
            for snapshot in sorted(snapshots_dir.iterdir()):
                if snapshot.is_dir():
                    found.append(snapshot)
                    print(snapshot)

if not found:
    print("Nessuno snapshot individuato nei percorsi controllati.")

BASE MODEL: google/medgemma-1.5-4b-it
REVISION IN ADAPTER CONFIG: None

FILE CONGELATI:
adapter_config.json 1465 365080625b497fee9f2b7e646df7f8ad0388defbe44840dfa67c276527d3fc50
adapter_model.safetensors 119280712 e64c995b6f3b42be3d9ff19c85d83c688cc6458673024dc9bcef0bf95d241098
chat_template.jinja 1532 7de1c58e208eda46e9c7f86397df37ec49883aeece39fb961e0a6b24088dd3c4
processor_config.json 519 90eeb521cab526796d26f85f6a9bcfb05128a71b7c74a8621205c37568aa3048
tokenizer.json 33384567 daab2354f8a74e70d70b4d1f804939b68a8c9624dd06cb7858e52dd8970e9726
tokenizer_config.json 1155445 3895ac44f8b9788247b6a66f26f2b2a3866a7ec2ed9f18bf32c12ce0bb410024
checkpoint-1245/trainer_state.json 376914 bdde58f1905e4489b03ef3719ba7112de741aaa3124b5dd95782502e3b600366

SNAPSHOT LOCALI DEL MODELLO BASE:
/mnt/f/hf_home_cache/hub/models--google--medgemma-1.5-4b-it/snapshots/91850547d9f0b2fdd21aa7c5f4f3d1a8a52c243b
/mnt/f/hf_home_cache/hub/models--google--medgemma-1.5-4b-it/snapshots/b05b6fa90147b76639de6522a843ff1eb

In [3]:
from pathlib import Path
from datetime import datetime
import hashlib

ADAPTER_DIR = Path(
    "/home/eugrug/.unsloth/studio/outputs/"
    "google_medgemma-1.5-4b-it__project-emcrag_ft_imb_010k_googlebase_v0702_1788538483"
)

SNAPSHOT_ROOTS = [
    Path("/mnt/f/hf_home_cache/hub/models--google--medgemma-1.5-4b-it"),
    Path("/mnt/f/hf_home_cache/models--google--medgemma-1.5-4b-it"),
    Path.home() / ".cache/huggingface/hub/models--google--medgemma-1.5-4b-it",
]

COMPARE_FILES = [
    "tokenizer.json",
    "tokenizer_config.json",
    "processor_config.json",
    "chat_template.jinja",
]

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

adapter_hashes = {
    name: sha256_file(ADAPTER_DIR / name)
    for name in COMPARE_FILES
    if (ADAPTER_DIR / name).is_file()
}

seen = set()

for model_root in SNAPSHOT_ROOTS:
    print(f"\nCACHE: {model_root}")

    ref_main = model_root / "refs" / "main"
    if ref_main.is_file():
        print("refs/main:", ref_main.read_text(encoding="utf-8").strip())
    else:
        print("refs/main: assente")

    snapshots_dir = model_root / "snapshots"
    if not snapshots_dir.is_dir():
        continue

    for snapshot in sorted(snapshots_dir.iterdir()):
        if not snapshot.is_dir():
            continue

        revision = snapshot.name
        key = (str(model_root), revision)
        if key in seen:
            continue
        seen.add(key)

        modified = datetime.fromtimestamp(
            snapshot.stat().st_mtime
        ).isoformat(timespec="seconds")

        print(f"\nREVISIONE: {revision}")
        print("Directory modificata:", modified)

        for name, adapter_hash in adapter_hashes.items():
            source_file = snapshot / name

            if not source_file.is_file():
                print(f"  {name}: ASSENTE")
                continue

            source_hash = sha256_file(source_file)
            status = "MATCH" if source_hash == adapter_hash else "DIVERSO"
            print(f"  {name}: {status} | {source_hash}")


CACHE: /mnt/f/hf_home_cache/hub/models--google--medgemma-1.5-4b-it
refs/main: 91850547d9f0b2fdd21aa7c5f4f3d1a8a52c243b

REVISIONE: 91850547d9f0b2fdd21aa7c5f4f3d1a8a52c243b
Directory modificata: 2026-04-16T12:20:32
  tokenizer.json: DIVERSO | 7d4046bf0505a327dd5a0abbb427ecd4fc82f99c2ceaa170bc61ecde12809b0c
  tokenizer_config.json: DIVERSO | 88269475414d45d3e59892d3519b0d4e28ec6416fecf6b523fb986cbe452075f
  processor_config.json: DIVERSO | 3ffd5f11778dc73e2b69b3c00535e4121e1badf7018136263cd17b5b34fbaa53
  chat_template.jinja: MATCH | 7de1c58e208eda46e9c7f86397df37ec49883aeece39fb961e0a6b24088dd3c4

REVISIONE: b05b6fa90147b76639de6522a843ff1ebd8dd832
Directory modificata: 2026-01-19T10:32:25
  tokenizer.json: DIVERSO | 7d4046bf0505a327dd5a0abbb427ecd4fc82f99c2ceaa170bc61ecde12809b0c
  tokenizer_config.json: DIVERSO | 88269475414d45d3e59892d3519b0d4e28ec6416fecf6b523fb986cbe452075f
  processor_config.json: DIVERSO | 3ffd5f11778dc73e2b69b3c00535e4121e1badf7018136263cd17b5b34fbaa53
  chat_t

## FT-10K — congelamento definitivo e provenienza del modello

### Motivazione del nuovo training

I precedenti esperimenti di fine-tuning erano stati eseguiti partendo da una
distribuzione prequantizzata pubblicata da Unsloth. Tale base non coincideva
formalmente con il repository utilizzato dal generatore della pipeline
EMCRAG v06.11.

Questo avrebbe introdotto un fattore confondente nel confronto sperimentale:
le differenze osservate avrebbero potuto dipendere sia dal fine-tuning sia
dalla diversa distribuzione del modello base.

Il run definitivo `EMCRAG_FT_IMB_010K_GOOGLEBASE_v0702` è stato pertanto
rieseguito partendo direttamente da:

`google/medgemma-1.5-4b-it`

La quantizzazione a 4 bit è stata utilizzata esclusivamente come tecnica
computazionale QLoRA durante il training. Non è stata utilizzata come base
la precedente distribuzione prequantizzata Unsloth.

Il nuovo `adapter_config.json` conferma:

`base_model_name_or_path = google/medgemma-1.5-4b-it`

Di conseguenza:

- M0 utilizza `google/medgemma-1.5-4b-it` senza adapter;
- M1 utilizza lo stesso modello base con il nuovo adapter FT-10K;
- R0 utilizza EMCRAG v06.11 con il modello Google-base nel generatore finale;
- R1 utilizzerà la stessa pipeline applicando il nuovo adapter esclusivamente
  al generatore finale.

La presenza dell’adapter FT-10K costituisce quindi la variabile sperimentale
principale nei confronti M0→M1 e R0→R1.

### Risultato del training definitivo

- Training run: `EMCRAG_FT_IMB_010K_GOOGLEBASE_v0702`
- Base model: `google/medgemma-1.5-4b-it`
- Fine-tuning method: QLoRA, caricamento a 4 bit
- Final checkpoint: `checkpoint-1245`
- Global step: 1245/1245
- Epoch: 1.0
- Final validation loss: `1.7569763660430908`
- Best-model automatic selection: not enabled

Validation-loss trajectory:

- step 312: `1.8491053581237793`
- step 624: `1.800355315208435`
- step 936: `1.769695520401001`
- step 1245: `1.7569763660430908`

La validation loss è diminuita a ogni valutazione. Il precedente valore
`1.756805301` apparteneva al vecchio training basato sulla distribuzione
Unsloth e non viene utilizzato per il run definitivo.

### Revisione del modello base

- Resolved Hugging Face revision:
  `91850547d9f0b2fdd21aa7c5f4f3d1a8a52c243b`
- Revision explicitly pinned during training: no
- Revision reconstructed from the local Hugging Face cache: yes

Tutte le cache disponibili associavano concordemente `refs/main` al commit
sopra indicato. Le altre revisioni presenti erano snapshot storici non
selezionati da `refs/main`. La revisione viene congelata ora per assicurare
la riproducibilità delle successive esecuzioni.

### Checksum degli artefatti definitivi

- `adapter_model.safetensors`  
  SHA256: `e64c995b6f3b42be3d9ff19c85d83c688cc6458673024dc9bcef0bf95d241098`

- `adapter_config.json`  
  SHA256: `365080625b497fee9f2b7e646df7f8ad0388defbe44840dfa67c276527d3fc50`

- `trainer_state.json`  
  SHA256: `bdde58f1905e4489b03ef3719ba7112de741aaa3124b5dd95782502e3b600366`

- `tokenizer.json`  
  SHA256: `daab2354f8a74e70d70b4d1f804939b68a8c9624dd06cb7858e52dd8970e9726`

- `tokenizer_config.json`  
  SHA256: `3895ac44f8b9788247b6a66f26f2b2a3866a7ec2ed9f18bf32c12ce0bb410024`

- `processor_config.json`  
  SHA256: `90eeb521cab526796d26f85f6a9bcfb05128a71b7c74a8621205c37568aa3048`

- `chat_template.jinja`  
  SHA256: `7de1c58e208eda46e9c7f86397df37ec49883aeece39fb961e0a6b24088dd3c4`

### Stato

Il FT-10K Google-base è definitivamente congelato. Non sono previsti
ulteriori training.